# 02. Composición de cartera semanal por fondo (CNV)

Baja la cartera de cada fondo money market, una por semana, desde la ficha del fondo en la CNV.

- **Fuente:** `https://www.cnv.gov.ar/SitioWeb/FondosComunesInversion/DetallesFCI/{id}` con `compo=true&fecha=AAAA-MM-DD`. Es lo mismo que hace el selector "Composición de carteras anteriores" de la ficha.
- **Unidad:** fondo × fecha de cartera. La cartera es una sola para todas las clases del fondo.
- **Contenido:** % del patrimonio en Disponibilidades (un solo número, sin banco), plazo fijo por banco, caución por mercado, títulos, cuotapartes de otros fondos, pasivos, y el patrimonio en pesos.
- **Salidas:**
  - `Data/raw/FondosCNV/carteras/<id>/<fecha>.html.gz` y `manifest_carteras.csv`;
  - `Data/crosswalks/fondos_cnv_lista.csv`, la lista de fondos de la CNV con su id;
  - `Data/crosswalks/fondos_cnv_lfc.csv`, el crosswalk entre el código CNV de la planilla y el id de la ficha, **a revisar a mano**;
  - `Data/processed/FondosCNV/carteras_encabezado.parquet` y `carteras_lineas.parquet`.

Requiere haber corrido el 01 (usa `planillas_mm.parquet` para saber qué fondos son money market).

In [ ]:
import sys, importlib
from pathlib import Path
import numpy as np
import pandas as pd

# el módulo vive en esta misma carpeta
sys.path.insert(0, str(Path.cwd()))
import fondoscnv as fc
importlib.reload(fc)

R = fc.rutas()          # raíz del repo y carpetas de datos
pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 40)
{k: str(v) for k, v in R.items()}

## 1. Lista de fondos de la CNV

Sale del buscador de la página principal de FCI. Cada fondo tiene un id interno (`lfc`) que **no** es el código CNV de la planilla (Mercado Fondo: 956 en la planilla, 63968 en la ficha). Es probable que la lista no incluya fondos ya liquidados; se chequea en el paso 2.

In [ ]:
s = fc.sesion()
lista = fc.lista_fondos_cnv(s)
lista.to_csv(R["xwalk"] / "fondos_cnv_lista.csv", index=False)
print(f"fondos en el buscador: {len(lista):,}")
lista.head()

## 2. Crosswalk planilla ↔ ficha

Para cada fondo money market que aparece alguna vez en las planillas se propone el id de la ficha por similitud de nombre, con los tres mejores candidatos. Se revisa a mano: se corrige `lfc` si hace falta, se pone `revisado = 1` y se anota cualquier cosa rara en `nota` (cambio de nombre, de gerente, fondo sin ficha). Si el archivo ya existe, las filas revisadas se conservan.

In [ ]:
mm = pd.read_parquet(R["proc"] / "planillas_mm.parquet")
fondos_mm = (mm.sort_values("fecha")
               .groupby("cod_cnv")
               .agg(fondo=("fondo", "last"), gerente=("gerente", "last"),
                    primera=("fecha", "min"), ultima=("fecha", "max"),
                    pn_max=("pn", "max"))
               .reset_index())
print(f"fondos money market ARS en las planillas: {len(fondos_mm)}")

prop = fc.proponer_crosswalk(fondos_mm, lista)
prop = prop.merge(fondos_mm[["cod_cnv", "primera", "ultima", "pn_max"]], on="cod_cnv", how="left")

ruta_xw = R["xwalk"] / "fondos_cnv_lfc.csv"
if ruta_xw.exists():
    viejo = pd.read_csv(ruta_xw)
    rev = viejo[viejo["revisado"] == 1]
    prop = pd.concat([rev, prop[~prop["cod_cnv"].isin(rev["cod_cnv"])]], ignore_index=True)
prop = prop.sort_values("pn_max", ascending=False)
prop.to_csv(ruta_xw, index=False)
print("revisados:", int(prop["revisado"].sum()), "| exactos:", int(prop["exacto"].sum()))
prop[["cod_cnv", "fondo_planilla", "lfc", "nombre_cnv", "score", "exacto", "revisado"]].head(25)

In [ ]:
# los que más conviene mirar: score bajo o fondos grandes sin match exacto
prop[(prop["score"] < 0.85) | ((prop["exacto"] == 0) & (prop["pn_max"] > 1e11))][
    ["cod_cnv", "fondo_planilla", "candidatos", "pn_max", "ultima"]]

## 3. Descarga

Una consulta por fondo y por viernes. La ficha devuelve la cartera vigente a esa fecha, con su fecha efectiva en "Cartera al". Unos 70 fondos × ~400 semanas son ~28.000 pedidos livianos: con pausa de 0,4 s, cerca de 3 horas. Conviene correr primero el piloto.

In [ ]:
MODO = "piloto"             # "piloto" o "completo"
SOLO_REVISADOS = False      # en "completo" conviene True, una vez revisado el crosswalk

xw = pd.read_csv(ruta_xw)
if MODO == "piloto":
    lfcs = xw.sort_values("pn_max", ascending=False)["lfc"].astype(str).head(5).tolist()
    fechas = fc.viernes("2025-05-02", "2025-09-26")
else:
    base = xw[xw["revisado"] == 1] if SOLO_REVISADOS else xw
    lfcs = base["lfc"].astype(str).drop_duplicates().tolist()
    fechas = fc.viernes("2019-01-04")
print(f"{len(lfcs)} fondos x {len(fechas)} semanas = {len(lfcs) * len(fechas):,} pedidos (menos los ya bajados)")

In [ ]:
n = fc.descargar_carteras(lfcs, fechas, R["raw_carteras"], R["raw"] / "manifest_carteras.csv",
                         pausa=0.4, s=s)
print("bajadas en esta corrida:", n)
man = pd.read_csv(R["raw"] / "manifest_carteras.csv")
man["estado"].value_counts()

## 4. Lectura y chequeos

Cada HTML se convierte en un encabezado (fondo, fecha de cartera, gerente, patrimonio) y en líneas (rubro, línea, tipo, contraparte, %). Las líneas de plazo fijo quedan con el nombre del banco tal como lo escribe la CNV ("Bco Nacion", "Nvo Bco Indus"). El pasaje a código de entidad BCRA queda para después.

In [ ]:
E, L = fc.apilar_carteras(R["raw_carteras"])
E.to_parquet(R["proc"] / "carteras_encabezado.parquet", index=False)
L.to_parquet(R["proc"] / "carteras_lineas.parquet", index=False)
print(f"páginas leídas: {len(E):,} | con cartera: {E['cartera_al'].notna().sum():,} | líneas: {len(L):,}")
print("carteras distintas (lfc, cartera_al):", E.dropna(subset=["cartera_al"])[["lfc", "cartera_al"]].drop_duplicates().shape[0])

In [ ]:
# 1) el id de la ficha puede cambiar de fondo en el tiempo: en 2019 el 63968 tenía
#    una cartera que no era de money market. Se listan los nombres por id.
nombres = (E.dropna(subset=["cartera_al"])
             .groupby(["lfc", "fondo_cnv"])["cartera_al"].agg(["min", "max", "count"])
             .reset_index())
cambian = nombres.groupby("lfc").filter(lambda g: len(g) > 1)
print("ids con más de un nombre de fondo:", cambian["lfc"].nunique())
cambian

In [ ]:
# 2) la suma de rubros debería dar 100; 3) tipos de línea que aparecen
E["total_pct"].describe()

In [ ]:
L[L["nivel"] == "detalle"].groupby(["rubro", "tipo"]).size().sort_values(ascending=False).head(20)

In [ ]:
# contrapartes de plazo fijo tal como vienen (insumo del crosswalk de bancos, más adelante)
(L[L["tipo"].isin(["plazo_fijo", "plazo_fijo_uva"])]
   .groupby("contraparte")["lfc"].nunique().sort_values(ascending=False)
   .rename("fondos").to_frame().head(40))